# Stage 2 — LSTM Baseline

Goal: a from-scratch LSTM sentiment classifier in PyTorch, trained on the
coffee competitive-set reviews. This is the baseline the transformer must beat.


2.4 Text to tensors (Dataset + DataLoader)
2.5 The model
2.6 Training loop
2.7 Evaluation

## 2.1 Importing libraries

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
from collections import Counter

## 2.2 Setup and data loading

In [4]:
ROOT = Path.cwd().parents[0]
train = pd.read_parquet(ROOT / "data/processed/coffee_train.parquet")

print(train.shape)
print(train["label"].value_counts(normalize=True).round(3))

(788195, 13)
label
1    0.865
0    0.135
Name: proportion, dtype: float64


## 2.3 Vocabulary

In [9]:
def build_vocab(texts, min_freq=2):
    # texts: list of review strings from the TRAINING set only
    # min_freq: a word must appear at least this many times to earn its own number
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())   # lowercase, split on spaces

    # two special markers first: padding and unknown
    word_to_idx = {"<pad>": 0, "<unk>": 1}

    for word, count in counter.items():
        if count >= min_freq:
            word_to_idx[word] = len(word_to_idx)   # next free number

    idx_to_word = {idx: word for word, idx in word_to_idx.items()}
    return word_to_idx, idx_to_word


def text_to_ids(text, word_to_idx):
    ids = []
    for word in text.lower().split():
        if word in word_to_idx:
            ids.append(word_to_idx[word])          # known word -> its number
        else:
            ids.append(word_to_idx["<unk>"])       # unknown word -> 1
    return ids


word_to_idx, idx_to_word = build_vocab(train["full_text"].tolist(), min_freq=20)

print("vocabulary size:", len(word_to_idx))
print("first few words:", list(word_to_idx.items())[:8])

vocabulary size: 21309
first few words: [('<pad>', 0), ('<unk>', 1), ('not', 2), ('actually', 3), ('for', 4), ('use', 5), ('in', 6), ('espresso', 7)]


In [ ]:
# counter = Counter()
# for text in train["full_text"].tolist():
#     counter.update(text.lower().split())

# for mf in [2, 3, 5, 10, 20]:
#     size = sum(1 for w, c in counter.items() if c >= mf) + 2  # +2 for <pad>, <unk>
#     print(f"min_freq={mf:>2} -> vocab size {size}")

min_freq= 2 -> vocab size 113703
min_freq= 3 -> vocab size 77783
min_freq= 5 -> vocab size 52686
min_freq=10 -> vocab size 33128
min_freq=20 -> vocab size 21309


In [ ]:
# total_tokens = sum(counter.values())   # every word occurrence, counted with repeats

# print(f"total tokens: {total_tokens:,}")
# print(f"total unique types: {len(counter):,}\n")

# for mf in [2, 3, 5, 10, 20, 50]:
#     kept_types = [w for w, c in counter.items() if c >= mf]
#     kept_tokens = sum(counter[w] for w in kept_types)
#     coverage = kept_tokens / total_tokens
#     print(f"min_freq={mf:>2}  vocab {len(kept_types)+2:>7,}  token coverage {coverage:.4f}")

total tokens: 24,409,386
total unique types: 307,161

min_freq= 2  vocab 113,703  token coverage 0.9921
min_freq= 3  vocab  77,783  token coverage 0.9891
min_freq= 5  vocab  52,686  token coverage 0.9857
min_freq=10  vocab  33,128  token coverage 0.9805
min_freq=20  vocab  21,309  token coverage 0.9739
min_freq=50  vocab  12,011  token coverage 0.9621
